# $$\text{Hyper Parameter Tuning}$$

Using *Exercise 2 in Machine Learning*, I will now present a way to tune hyper parameters. For this I will use **Optuna**, an open source optimization framework [https://optuna.org/].

# Preliminaries

In [3]:
# Standard python modules
import numpy as np
import matplotlib.pyplot as plt

# Machine learning module
import torch
import torch.optim as optim 
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Hyper-Parameter module
import optuna

Last time I did the exercise using PyTorch, i figured that GPU-acceleration was slower than using the CPU. This is mostly due to the exercises simple nature, but it might also be due to my CPU (at home) being pretty beefy. It might not be the case if we were to run the same code in a cluster, but it might be hard to actually figure out due to the code being so fast to run.

In [5]:
device = torch.device(device='cpu') # CPU is selected by default

<h2 align='center'>1 Loading the data</h2>

In [6]:
data = np.loadtxt("Data_Exercise_2/data.txt", delimiter=',')

<h2 style='text-align: center;'>2 Splitting the data</h2>

In [10]:
def split(data):
    X = data[:, 0:3]
    Y = data[:, 3:]
    return X, Y

In [8]:
X_raw, Y_raw = split(data)

<h2 style='text-align: center;'>3 Defining training- and validation data</h2>

To prevent the split to simply define the first 80% as training data, and the last 20% as validation data, we define ```shuffle=True```. This makes sure the training- and validation data is randomly selected. 

To make sure we get the same data set each training run, we enable ```random_state=1```, which fixes the randomness.

In [9]:
X_train_raw, X_val_raw, Y_train_raw, Y_val_raw = train_test_split(
    X_raw, Y_raw, test_size=0.2, random_state=1, shuffle=True
)

<h2 style='text-align: center;'>4 Converting data to tensors</h2>

Since I will be using PyTorch, we now have to change the data type from NumPy arrays to tensors.

In [11]:
def array_to_tensor(array):
    return torch.tensor(array, dtype=torch.float32).to(device)

<h2 style='text-align: center'> 5 Scaling the data</h2>

```fit_transform()``` does two operations at once:
1. ```fit()``` in combination with ```MinMaxScaler()``` finds the minimum and maximum value of each column
2. ```transform()``` learns the scaling parameters from the data.

In [13]:
X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()

In [14]:
scaled_X_train = X_scaler.fit_transform(X_train_raw)
scaled_X_val   = X_scaler.transform(X_val_raw)

In [15]:
scaled_Y_train = Y_scaler.fit_transform(Y_train_raw)
scaled_Y_val   = Y_scaler.transform(Y_val_raw)

In [16]:
training_X  = array_to_tensor(scaled_X_train)
val_X       = array_to_tensor(scaled_X_val)

training_Y  = array_to_tensor(scaled_Y_train)
val_Y       = array_to_tensor(scaled_Y_val)

<h2 style='text-align: center;'>Comment on Optuna</h2>

When we use Optuna, the main thing to watch is where we place our model, optimizer, dataloader and training loop. These should be carefully defined inside the Optuna objective function. Which is what we want to optimize. But we still need to define the ANN ass a class, meaning we define it outside the objective function. Thus, each trial needs to create a fresh model!